# Wanderbricks — Quick Forecast Lab

A scratchpad that mirrors the real pipeline steps so you can experiment before touching the job: Prophet baseline, then a quick XGBoost trial, all on one station's data.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "iraonfridays")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
print(f"target: {CATALOG}.{SCHEMA}")

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{name}")

for t in ["gsod_bronze", "gsod_silver", "weather_features", "forecasts"]:
    print(f"{t:20s} {'EXISTS' if table_exists(t) else 'MISSING - run the DLT pipeline first'}")

## 1. Prophet baseline (like `ml/04_baseline.py`)

In [ ]:
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error

if not table_exists("gsod_silver"):
    print("gsod_silver missing - run the DLT pipeline first")
else:
    station = dbutils.widgets.get("station").strip() or None
    if not station:
        station = (
            spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
            .groupBy("station").count().orderBy(F.desc("count"))
            .limit(1).collect()[0][0]
        )
    series = (
        spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
        .filter(F.col("station") == station)
        .select("date", "temp_c").filter("temp_c IS NOT NULL")
        .orderBy("date").toPandas()
        .rename(columns={"date": "ds", "temp_c": "y"})
    )
    horizon = 30
    train, test = series.iloc[:-horizon], series.iloc[-horizon:]
    model = Prophet(daily_seasonality=True).fit(train)
    fc = model.predict(test[["ds"]])
    rmse = float(np.sqrt(mean_squared_error(test["y"], fc["yhat"])))
    mae = float(mean_absolute_error(test["y"], fc["yhat"]))
    print(f"station={station} prophet rmse={rmse:.3f} mae={mae:.3f}")
    fig = model.plot(fc)
    plt.show()

## 2. Quick XGBoost trial (like `ml/05_train_xgb.py`)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

if table_exists("weather_features"):
    pdf = (
        spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
        .drop("date").toPandas().dropna()
    )
    y = pdf.pop("temp_c")
    X = pdf
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, shuffle=False)
    m = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, objective="reg:squarederror")
    m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    pred = m.predict(X_te)
    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    mae = float(mean_absolute_error(y_te, pred))
    print(f"xgb rmse={rmse:.3f} mae={mae:.3f}  (baseline bar is above)")
    importances = pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False)
    display(importances.head(10))

## 3. Run the real thing

When you are happy with experiments here, run the production chain from the CLI:

```sh
databricks bundle run wanderbricks_pipeline --refresh-all   # ingest + clean + features
databricks bundle run wanderbricks_job --refresh-all        # baseline + train + score
```

Any experiment you like here can be promoted into `ml/` or `pipelines/`.